Make NumPy available:

In [1]:
import numpy as np

## Exercise 07.1 (indexing and timing)

Create two very long NumPy 1D-arrays (vectors) `x` and `y` and sum the arrays using:

1. The NumPy addition syntax, `z = x + y`; and
2. A `for` loop that computes the sum entry-by-entry

Compare the time required for the two approaches for arrays of different lengths (use a very long array for 
the timing). The values of the array entries are not important for this test. Use `%time` to report the time.

*Hint:* To loop over an array using indices, try a construction like:

In [2]:
x = np.ones(10)
y = np.ones(len(x))
for i in range(len(x)):
    print(x[i]*y[i])

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


#### (1) Add two arrays using built-in addition operator:

In [3]:
x = np.ones(10000)
y = np.ones(len(x))
#work time for built-in operator
%time z = x + y

CPU times: total: 0 ns
Wall time: 61.5 μs


#### (2) Add two arrays using own implementation:

In [4]:
x = np.ones(10000)
y = np.ones(len(x))
z = np.ones(len(x))

def array_sum(x, y):
    z = np.ones(len(x))
    for i in range(len(x)): 
        z[i] = x[i] + y[i]
    return z


array_sum(x, y)
%time z = array_sum(x, y)

CPU times: total: 0 ns
Wall time: 1.16 ms


### Optional extension: just-in-time (JIT) compilation

You will see a large difference in the time required between your NumPy and 'plain' Python implementations. This is due to Python being an *interpreted* language as opposed to a *compiled* language. A way to speed up plain Python implementions is to convert the interpreted Python code into compiled code. A tool for doing this is [Numba](https://numba.pydata.org/).

Below is an example using Numba and JIT to accelerate a computation:

In [5]:
%pip -q install numba 
import numba
import math

def compute_sine_native(x):
    z = np.zeros(len(x))
    for i in range(len(z)):
        z[i] = math.sin(x[i])
    return z

@numba.jit
def compute_sine_jit(x):
    z = np.zeros(len(x))
    for i in range(len(z)):
        z[i] = math.sin(x[i])
    return z
    
x = np.ones(10000000)
%time z = compute_sine_native(x)
compute_sine_jit(x)
%time z = compute_sine_jit(x)

Note: you may need to restart the kernel to use updated packages.
CPU times: total: 797 ms
Wall time: 800 ms
CPU times: total: 141 ms
Wall time: 139 ms


**Task:** Test if Numba can be used to accelerate your implementation that uses indexing to sum two arrays, and by how much.

In [6]:
%pip -q install numba
import numba

#it was considerably faster (tens of micro seconds vs a few miliseconds)

@numba.jit
def array_sum(x, y):
    z = np.ones(len(x))
    for i in range(len(x)): 
        z[i] = x[i] + y[i]
    return z

x = np.ones(10000)
y = np.ones(len(x))

array_sum(x, y)
%time z = array_sum(x, y)

Note: you may need to restart the kernel to use updated packages.
CPU times: total: 0 ns
Wall time: 4.92 ms


## Exercise 07.2 (member functions and slicing)

Anonymised scores (out of 60) for an examination are stored in a NumPy array. Write:

1. A function that takes a NumPy array of the raw scores and returns the scores as percentages, sorted from 
   lowest to highest (try using `scores.sort()`, where `scores` is a NumPy array holding the scores).
1. A function that returns the maximum, minimum and mean of the raw scores as a dictionary with the 
   keys '`min`', '`max`' and '`mean`'. Use the NumPy array functions `min()`, `max()` and `mean()` to do the 
   computation, e.g. `max = scores.max()`.  
   
   Design your function for the min, max and mean to optionally exclude the highest and lowest scores from the 
   computation of the min, max and mean. 
   
   *Hint:* sort the array of scores and use array slicing to exclude
   the first and the last entries.

Use the scores 
```python
scores = np.array([58.0, 35.0, 24.0, 42, 7.8])
```
to test your functions.

In [7]:
def to_percentage_and_sort(scores):
    #calculate percentage scores
    percents = [0 for i in scores]
    for i, score in enumerate(scores):
        percents[i] = (score/60) * 100
    percents.sort()

    return percents

def statistics(scores, exclude=False):
    ordered = sorted(scores)
    #initialise dictionary
    stats = {}
    #consider exclusion of extremes
    sum = 0
    if exclude == False:
        stats["min"] = ordered[0]
        stats["max"] = ordered[-1]
        #calculate mean
        for score in ordered[0:]:
            sum += score
        stats["mean"] = sum/len(ordered)
    elif exclude == True:
        stats["min"] = ordered[1]
        stats["max"] = ordered[-2]
        #calculate mean
        for score in ordered[1 : -1]:
            sum += score
        stats["mean"] = sum/(len(ordered) - 2)

    return stats

#own tests
scores = ([58.0, 35.0, 24.0, 42, 7.8])
assert np.isclose(to_percentage_and_sort(scores), [ 13.0, 40.0, 58.33333333,  70.0, 96.66666667]).all()
s2 = statistics(scores)
assert np.isclose(s2["min"], 7.8)
assert np.isclose(s2["mean"], 33.36)
assert np.isclose(s2["max"], 58.0)

In [8]:
## tests ##

scores = np.array([35.0, 58.0, 24.0, 42, 7.8])
assert np.isclose(to_percentage_and_sort(scores), [ 13.0, 40.0, 58.33333333,  70.0, 96.66666667]).all()

s0 = statistics(scores)
assert np.isclose(s0["min"], 7.8)
assert np.isclose(s0["mean"], 33.36)
assert np.isclose(s0["max"], 58.0)

s1 = statistics(scores, True)
assert np.isclose(s1["min"], 24.0)
assert np.isclose(s1["mean"], 33.666666666666666667)
assert np.isclose(s1["max"], 42.0)

## Exercise 07.3 (slicing)

For the two-dimensional array

In [9]:
A = np.array([[4.0, 7.0, -2.43, 67.1],
             [-4.0, 64.0, 54.7, -3.33],
             [2.43, 23.2, 3.64, 4.11],
             [1.2, 2.5, -113.2, 323.22]])
print(A)

[[   4.      7.     -2.43   67.1 ]
 [  -4.     64.     54.7    -3.33]
 [   2.43   23.2     3.64    4.11]
 [   1.2     2.5  -113.2   323.22]]


use array slicing for the below operations, printing the results to the screen to check. Try to use array slicing such that your code would still work if the dimensions of `A` were enlarged.

**Check your results carefully against hand computations.**

#### 1. Extract the third column as a 1D array

In [10]:
column_3 = np.array(A[:, 2])
##test##
assert np.isclose(column_3, np.array([-2.43, 54.7, 3.64, -113.2])).all()

#### 2. Extract the first two rows as a 2D sub-array

In [11]:
two_rows = np.array(A[:2])
##test##
assert np.isclose(two_rows, np.array([[4.0, 7.0, -2.43, 67.1], [-4.0, 64.0, 54.7, -3.33]])).all()

#### 3.  Extract the bottom-right $2 \times 2$ block as a 2D sub-array

In [12]:
two_by_two = np.array(A[2:, 2:])
##test##
assert np.isclose(np.array(two_by_two), np.array([[3.64, 4.11], [-113.2, 323.22]])).all()

#### 4. Sum the last column

In [13]:
last_column = A[:, 3]
sum = 0
for item in last_column:
    sum += item
##test##
assert np.isclose(sum, 391.10)

#### Compute transpose

Compute the transpose of `A` (search online to find the function/syntax to do this).

In [14]:
Ainv = np.matrix.transpose(A)
##test##
assert (Ainv == np.array([[4.0, -4.0, 2.43, 1.2], [7.0, 64.0, 23.2, 2.5], [-2.43, 54.7, 3.64, -113.2], [67.1, -3.33, 4.11, 323.22]])).all()

## Exercise 07.4 (optional extension)

In a previous exercise (04.5) you implemented the bisection algorithm to find approximate roots of a mathematical function. Use the SciPy bisection function `optimize.bisect` (https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.bisect.html) to find roots of the mathematical function that was used in the previous exercise. Compare the results computed by SciPy and your program from the earlier exercise, and compare the computational time (using `%time`).

In [15]:
from scipy import optimize

def my_f(x):
    """Evaluate polynomial function"""
    return x**5 / 10 + x**3 - 10 * x**2 + 4 * x + 7

# My own function from exercise 4
def compute_root(f, x0, x1, tol, max_it):
    """Compute roots of a function using bisection"""
    error = tol + 1
    it = 0
    while error >= tol:
        #compute midpoint and the functions value at it
        x_mid = (x0 + x1)/2
        f_mid = f(x_mid)
        
        #check for position of the root and adjust range
        if f(x0) * f_mid < 0:
            x1 = x_mid
        else:
            x0 = x_mid
        
        error = abs(f_mid)
        it += 1

        #guard against infinite loop
        if it > max_it:
            print(f"Number of iterations has surpassed {max_it}, breaking out of loop...")
            break
     
    return x_mid, f_mid, it

## test
%time x, f, num_it = compute_root(my_f, x0=0, x1=2, tol=1.0e-6, max_it=1000)
%time optimize.bisect(f)
assert math.isclose(x, 1.1568354368209839)

ImportError: DLL load failed while importing _moduleTNC: An Application Control policy has blocked this file.